# Evaluate RAG with NeMo Retriever and RAGAS

Configure one block near the top, choose **Run All Cells**, and receive document-retrieval metrics at the end. The default path downloads FinanceBench when needed, builds or reuses a LanceDB index, retrieves evidence, and presents aggregate and per-question results. Answer generation and RAGAS judging are opt-in stages.

## Metrics

- **Document Recall@1, @3, @5, and @10** — the fraction of relevant source documents found by each unique-document cutoff.
- **Document nDCG@1, @3, @5, and @10** — rewards relevant source documents more when they appear earlier in the unique-document ranking.
- **Answer Accuracy** — compares the generated response with the reference answer.
- **Context Relevance** — measures whether the retrieved passages address the question.
- **Response Groundedness** — checks whether the response is supported by the retrieved passages.

> **First-time setup:** Run this notebook from the repository `examples` directory. Install dependencies once, restart the kernel, set the variables below, and then use **Run All Cells**. Local retrieval evaluation needs no API key; hosted embedding, generation, and judging do.


## Run Configuration

This is the only cell most users need to edit. `DATASET_MODE="financebench"` is ready to run. To use your own data, select `"custom"` and fill in `CUSTOM_DATASET_CONFIG`.

`INFERENCE_MODE="local"` runs NRL extraction and embedding on local GPUs; `"hosted"` sends Page Elements, OCR, and embedding to NVIDIA endpoints and requires no GPU. Local and hosted indexes are kept separate. A missing index is always built, while `OVERWRITE_INDEX=True` replaces an existing index. Generation and judging remain optional hosted stages.

To score rankings from VAST or another vector database, set `EXTERNAL_RESULTS_PATH` to its JSONL output. External mode skips NRL indexing and retrieval and uses the same metrics and optional answer evaluation as the reference path.


In [ ]:
from pathlib import Path

# Dataset: "financebench" or "custom"
DATASET_MODE = "financebench"
INFERENCE_MODE = "local"  # "local" or "hosted"

FINANCEBENCH_DIR = Path("../data/financebench")
CUSTOM_DATASET_CONFIG = {
    "name": "my_dataset",
    "corpus_dir": Path("../data/my_dataset/corpus"),
    "ground_truth_path": Path("../data/my_dataset/ground_truth.jsonl"),
    "ground_truth_format": "jsonl",  # csv, json, or jsonl
    "id_field": None,
    "query_field": "question",
    "answer_field": "answer",
    "document_field": None,
}

# Evaluation
MAX_QUESTIONS = 50  # None evaluates the complete ground-truth split.
OVERWRITE_INDEX = False  # Set True after changing the corpus or model.
EXTERNAL_RESULTS_PATH = None  # Or Path("../results/partner_results.jsonl").

# Optional hosted stages
RUN_GENERATION = False
RUN_JUDGING = False

# Models
EMBED_MODEL = {
    "local": "nvidia/Nemotron-3-Embed-1B-BF16",
    "hosted": "nvidia/nemotron-3-embed-1b",
}[INFERENCE_MODE]
GENERATOR_MODEL = "nvidia/nemotron-3-super-120b-a12b"
JUDGE_MODEL = "nvidia/nemotron-3-super-120b-a12b"

assert DATASET_MODE in {"financebench", "custom"}
assert INFERENCE_MODE in {"local", "hosted"}
assert MAX_QUESTIONS is None or MAX_QUESTIONS > 0


## 1. Install Dependencies

Install the local GPU dependencies when `INFERENCE_MODE="local"` and NRL performs retrieval. Hosted and external-results modes intentionally install only the base NeMo Retriever package: they do not declare PyTorch, Transformers, vLLM, Triton, or local Nemotron model packages. First-time local setup downloads a larger CUDA/model stack and can take several minutes.

Run this cell once, restart the kernel, and then use **Run All Cells**. The `ipykernel<7` requirement avoids a `nest_asyncio` context collision in RAGAS 0.3.2.


In [ ]:
USE_LOCAL_NRL = INFERENCE_MODE == "local" and EXTERNAL_RESULTS_PATH is None
NRL_INSTALL_TARGET = (
    "../nemo_retriever[local]" if USE_LOCAL_NRL else "../nemo_retriever"
)
%pip install -qU -e "$NRL_INSTALL_TARGET" "ragas==0.3.2" "datasets>=4.8.2" "huggingface-hub>=1.5,<2" "ipykernel<7"


## 2. Prepare the Dataset

FinanceBench is downloaded only when its directory is absent. Custom datasets require a supported document corpus and a CSV, JSON, or JSONL ground-truth file.

For document-level metrics, `document_field` may contain one identifier or a list of relevant identifiers. Each value must normalize to the identifier returned by Retriever metadata (`source_id`, `source`, `path`, or `source_path`): matching is case-insensitive after removing directories and the final file extension. For example, `reports/ACME_2024.pdf` matches `ACME_2024.pdf`. Set `document_field=None` when this mapping is unavailable; retrieval metrics will be skipped.


In [ ]:
import subprocess

if DATASET_MODE == "financebench":
    if not FINANCEBENCH_DIR.exists():
        print(f"Downloading FinanceBench to {FINANCEBENCH_DIR}...")
        FINANCEBENCH_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/patronus-ai/financebench.git",
                str(FINANCEBENCH_DIR),
            ],
            check=True,
        )
    else:
        print(f"Reusing FinanceBench at {FINANCEBENCH_DIR}.")

    dataset_config = {
        "name": "financebench",
        "corpus_dir": FINANCEBENCH_DIR / "pdfs",
        "ground_truth_path": (
            FINANCEBENCH_DIR
            / "data"
            / "financebench_open_source.jsonl"
        ),
        "ground_truth_format": "jsonl",
        "id_field": "financebench_id",
        "query_field": "question",
        "answer_field": "answer",
        "document_field": "doc_name",
    }
else:
    dataset_config = dict(CUSTOM_DATASET_CONFIG)
    dataset_config["corpus_dir"] = Path(dataset_config["corpus_dir"])
    dataset_config["ground_truth_path"] = Path(
        dataset_config["ground_truth_path"]
    )

LANCEDB_URI = f"lancedb-{dataset_config['name']}"
if INFERENCE_MODE == "local":
    LANCEDB_URI += "-local"
TABLE_NAME = dataset_config["name"]


## 3. Validate the Dataset

Validate paths, load ground-truth records, check the configured fields, and print a compact runtime manifest before starting expensive work.


In [ ]:
import csv
import json

corpus_dir = Path(dataset_config["corpus_dir"])
ground_truth_path = Path(dataset_config["ground_truth_path"])
external_results_path = (
    Path(EXTERNAL_RESULTS_PATH)
    if EXTERNAL_RESULTS_PATH is not None
    else None
)
ground_truth_format = dataset_config["ground_truth_format"].lower()

assert corpus_dir.is_dir(), f"Corpus directory not found: {corpus_dir}"
assert ground_truth_path.is_file(), (
    f"Ground-truth file not found: {ground_truth_path}"
)
if external_results_path is not None:
    assert external_results_path.is_file(), (
        f"External retrieval results not found: {external_results_path}"
    )

corpus_files = sorted(path for path in corpus_dir.rglob("*") if path.is_file())
assert corpus_files, f"No files found in corpus directory: {corpus_dir}"

if ground_truth_format == "jsonl":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = [
            json.loads(line) for line in file if line.strip()
        ]
elif ground_truth_format == "json":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = json.load(file)
    assert isinstance(ground_truth_records, list), (
        "Ground-truth JSON must contain a list of records."
    )
elif ground_truth_format == "csv":
    with ground_truth_path.open(encoding="utf-8", newline="") as file:
        ground_truth_records = list(csv.DictReader(file))
else:
    raise ValueError("ground_truth_format must be 'csv', 'json', or 'jsonl'.")

assert ground_truth_records, f"No records found in {ground_truth_path}"

configured_fields = {
    name: dataset_config.get(name)
    for name in ("id_field", "query_field", "answer_field", "document_field")
    if dataset_config.get(name)
}
missing_fields = {
    field
    for field in configured_fields.values()
    if any(field not in record for record in ground_truth_records)
}
assert not missing_fields, (
    f"Missing configured ground-truth fields: {sorted(missing_fields)}"
)

query_field = dataset_config["query_field"]
answer_field = dataset_config["answer_field"]
assert all(
    str(record[query_field]).strip() for record in ground_truth_records
), "Found an empty question."
assert all(
    str(record[answer_field]).strip() for record in ground_truth_records
), "Found an empty reference answer."

print(f"Dataset: {dataset_config['name']}")
print(f"Corpus files: {len(corpus_files)}")
print(f"Ground-truth records: {len(ground_truth_records)}")
print(f"Example question: {ground_truth_records[0][query_field]}")

import importlib.metadata as package_metadata
import platform


def git_revision(path):
    result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=path,
        capture_output=True,
        check=False,
        text=True,
    )
    return result.stdout.strip() or "unavailable"


try:
    nrl_version = package_metadata.version("nemo-retriever")
except package_metadata.PackageNotFoundError:
    nrl_version = "unavailable"

runtime_manifest = {
    "retrieval_source": (
        "external" if external_results_path is not None else "nrl"
    ),
    "inference_mode": INFERENCE_MODE,
    "nemo_retriever_version": nrl_version,
    "nrl_git_revision": git_revision(Path("../nemo_retriever")),
    "python_version": platform.python_version(),
    "embedding_model": (
        None if external_results_path is not None else EMBED_MODEL
    ),
    "dataset_revision": (
        git_revision(FINANCEBENCH_DIR)
        if DATASET_MODE == "financebench"
        else "custom"
    ),
}

if USE_LOCAL_NRL:
    try:
        import torch

        runtime_manifest.update(
            {
                "pytorch_version": torch.__version__,
                "cuda_version": torch.version.cuda,
                "gpu_count": torch.cuda.device_count(),
                "gpus": [
                    torch.cuda.get_device_name(index)
                    for index in range(torch.cuda.device_count())
                ],
            }
        )
        from nemo_retriever.models.hf_model_registry import get_hf_revision

        runtime_manifest["embedding_model_revision"] = get_hf_revision(
            EMBED_MODEL
        )
    except Exception as error:
        runtime_manifest["local_runtime_error"] = (
            f"{type(error).__name__}: {error}"
        )

print("Runtime manifest:")
print(json.dumps(runtime_manifest, indent=2))

if USE_LOCAL_NRL:
    assert "local_runtime_error" not in runtime_manifest, (
        "Local inference setup failed: "
        + runtime_manifest["local_runtime_error"]
    )
    assert runtime_manifest["gpu_count"] > 0, (
        "Local inference requires a CUDA-enabled PyTorch installation and "
        "at least one visible NVIDIA GPU."
    )


## 4. Build or Reuse the LanceDB Index

The notebook always ingests when the selected mode’s table is missing. When it exists, `OVERWRITE_INDEX=False` reuses it and `True` rebuilds it with `--overwrite`.

Local mode runs NRL extraction and Nemotron 3 embedding on the available GPUs. Hosted mode sends rendered document pages or crops to NVIDIA’s Page Elements and OCR services, and extracted text to the embedding service. It requires `NVIDIA_API_KEY` but does not require a GPU. Do not use hosted mode for corpora that must remain within your environment. NRL’s batch planner chooses concurrency and GPU placement; this notebook does not override worker counts.

> **Ingestion can take a while:** a large corpus may take tens of minutes or hours depending on extraction, model startup, endpoint speed, and hardware. The batch CLI streams its available progress below.


In [ ]:
import os
import shutil
import subprocess
import sys
from getpass import getpass

PAGE_ELEMENTS_INVOKE_URL = (
    "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3"
)
OCR_INVOKE_URL = (
    "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-ocr-v2"
)
EMBED_INVOKE_URL = "https://integrate.api.nvidia.com/v1/embeddings"


def require_nvidia_api_key():
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key: ")
    return os.environ["NVIDIA_API_KEY"]


os.environ["PATH"] = (
    os.path.dirname(sys.executable)
    + os.pathsep
    + os.environ["PATH"]
)

database = None
if external_results_path is not None:
    print(
        "External retrieval results supplied; skipping NRL indexing: "
        f"{external_results_path}"
    )
else:
    import lancedb

    database = lancedb.connect(LANCEDB_URI)
    table_exists = TABLE_NAME in set(database.list_tables(limit=None).tables)
    should_ingest = OVERWRITE_INDEX or not table_exists

    if should_ingest:
        reason = "OVERWRITE_INDEX=True" if table_exists else "table is missing"
        print(
            f"Starting ingestion because {reason}: "
            f"{LANCEDB_URI}/{TABLE_NAME}"
        )

        retriever_executable = shutil.which("retriever")
        assert retriever_executable, (
            "The retriever CLI was not found. Run the dependency cell first."
        )
        ingest_command = [
            retriever_executable,
            "ingest",
            "batch",
            str(corpus_dir),
            "--lancedb-uri",
            str(LANCEDB_URI),
            "--table-name",
            str(TABLE_NAME),
            "--embed-model-name",
            EMBED_MODEL,
        ]
        if INFERENCE_MODE == "local":
            ingest_command.extend(
                ["--local-ingest-embed-backend", "hf"]
            )
        else:
            require_nvidia_api_key()
            ingest_command.extend(
                [
                    "--page-elements-invoke-url",
                    PAGE_ELEMENTS_INVOKE_URL,
                    "--ocr-invoke-url",
                    OCR_INVOKE_URL,
                    "--embed-invoke-url",
                    EMBED_INVOKE_URL,
                ]
            )
        if table_exists:
            ingest_command.append("--overwrite")
        subprocess.run(ingest_command, check=True)

        database = lancedb.connect(LANCEDB_URI)
        table_exists = TABLE_NAME in set(
            database.list_tables(limit=None).tables
        )
        assert table_exists, (
            f"Ingestion completed without creating {LANCEDB_URI}/{TABLE_NAME}."
        )
    else:
        print(f"Reusing existing table: {LANCEDB_URI}/{TABLE_NAME}")


## 5. Retrieve Contexts and Measure Document Retrieval

With `EXTERNAL_RESULTS_PATH=None`, Retriever embeds questions using the same local or hosted mode used to build the index. Local mode may pause while the embedding model first loads. Retrieval starts with 100 chunk candidates and expands to at most 1,000. It stops after finding the smaller of 10 unique documents or the known corpus-document count. If no recognized source identifier is present for a query, validation stops immediately instead of searching the complete chunk index.

For a partner database, provide one JSON object per line. `query_id` must match the configured ground-truth ID, and `results` must be ordered from most to least relevant. Each result needs a `document_id` for retrieval metrics and its retrieved `text` for optional generation and judging:

```json
{"query_id":"financebench_id_03882","results":[{"document_id":"report.pdf","text":"Retrieved passage..."}]}
```

Both paths deduplicate documents by normalized identifier **before** applying the required 1, 3, 5, and 10 cutoffs. Ground-truth identifiers are treated as binary relevance labels.


In [ ]:
import math

import pandas as pd

RETRIEVAL_CUTOFFS = (1, 3, 5, 10)
INITIAL_RETRIEVAL_CANDIDATES = 100
MAX_RETRIEVAL_CANDIDATES = 1_000
GENERATION_CONTEXT_CHUNKS = 10


def normalise_document_name(value):
    """Normalize a ground-truth or retrieved document identifier."""
    text = str(value or "").strip()
    if not text:
        return ""
    return Path(text).stem.casefold()


def source_document_name(metadata):
    """Extract the source document name from retrieval metadata."""
    for key in (
        "document_id",
        "source_id",
        "source",
        "path",
        "source_path",
    ):
        value = metadata.get(key)
        if not value:
            continue

        if isinstance(value, dict):
            nested = source_document_name(value)
            if nested:
                return nested

        if isinstance(value, str) and value.lstrip().startswith("{"):
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError:
                parsed = None
            if isinstance(parsed, dict):
                nested = source_document_name(parsed)
                if nested:
                    return nested

        return normalise_document_name(value)
    return ""


def ranked_unique_documents(metadata):
    """Collapse chunks to the first occurrence of each source document."""
    ranked = []
    seen = set()
    for hit in metadata:
        document = source_document_name(hit)
        if document and document not in seen:
            seen.add(document)
            ranked.append(document)
    return ranked


def gold_document_names(value):
    values = value if isinstance(value, (list, tuple, set)) else [value]
    return {
        normalised
        for item in values
        if (normalised := normalise_document_name(item))
    }


def document_recall_at_k(ranked_documents, gold_documents, k):
    gold = gold_document_names(gold_documents)
    if not gold:
        return None
    retrieved = set(ranked_documents[:k])
    return len(retrieved & gold) / len(gold)


def document_ndcg_at_k(ranked_documents, gold_documents, k):
    gold = gold_document_names(gold_documents)
    if not gold:
        return None
    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, document in enumerate(ranked_documents[:k], start=1)
        if document in gold
    )
    ideal_relevant = min(len(gold), k)
    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )
    return dcg / idcg if idcg else None


def record_query_id(record, position):
    value = record.get(id_field) if id_field else position
    return position if value is None else value


def validate_retrieval_metadata(results, query_ids_to_validate):
    missing_source_ids = [
        str(query_id)
        for query_id, result in zip(query_ids_to_validate, results)
        if not any(
            source_document_name(metadata)
            for metadata in result.metadata
        )
    ]
    if missing_source_ids:
        preview = missing_source_ids[:10]
        raise ValueError(
            "Retriever metadata contains no recognized source identifier "
            f"for {len(missing_source_ids)} queries: {preview}. Expected "
            "source_id, source, path, or source_path; document-level "
            "metrics and adaptive unique-document retrieval cannot continue."
        )


def load_external_results(path, expected_query_ids):
    by_query_id = {}
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            payload = json.loads(line)
            if not isinstance(payload, dict):
                raise ValueError(
                    f"External results line {line_number} must be an object."
                )
            if "query_id" not in payload or "results" not in payload:
                raise ValueError(
                    f"External results line {line_number} requires "
                    "query_id and results."
                )

            query_key = str(payload["query_id"])
            if query_key in by_query_id:
                raise ValueError(f"Duplicate external query_id: {query_key}")
            if not isinstance(payload["results"], list):
                raise ValueError(
                    f"results for query_id {query_key} must be a list."
                )

            hits = []
            for rank, hit in enumerate(payload["results"], start=1):
                if not isinstance(hit, dict):
                    raise ValueError(
                        f"Result {rank} for query_id {query_key} "
                        "must be an object."
                    )
                document_id = str(hit.get("document_id", "")).strip()
                text = hit.get("text")
                if not document_id:
                    raise ValueError(
                        f"Result {rank} for query_id {query_key} "
                        "has no document_id."
                    )
                if not isinstance(text, str) or not text.strip():
                    raise ValueError(
                        f"Result {rank} for query_id {query_key} "
                        "has no retrieved text."
                    )
                hits.append({"document_id": document_id, "text": text})
            by_query_id[query_key] = hits

    expected_keys = {str(query_id) for query_id in expected_query_ids}
    missing = expected_keys - by_query_id.keys()
    if missing:
        preview = sorted(missing)[:10]
        raise ValueError(
            f"External results are missing {len(missing)} selected query IDs: "
            f"{preview}"
        )
    extras = by_query_id.keys() - expected_keys
    if extras:
        print(f"Ignoring {len(extras)} external query IDs not selected for this run.")
    return by_query_id


selected_records = (
    ground_truth_records
    if MAX_QUESTIONS is None
    else ground_truth_records[:MAX_QUESTIONS]
)
id_field = dataset_config.get("id_field")
document_field = dataset_config.get("document_field")
questions = [record[query_field] for record in selected_records]
query_ids = [
    record_query_id(record, position)
    for position, record in enumerate(selected_records, start=1)
]
assert len({str(query_id) for query_id in query_ids}) == len(query_ids), (
    "Selected ground-truth query IDs must be unique."
)

target_unique_documents = (
    min(max(RETRIEVAL_CUTOFFS), len(corpus_files))
    if document_field
    else 0
)

if external_results_path is None:
    from nemo_retriever.graph.retriever import Retriever

    table_row_count = database.open_table(TABLE_NAME).count_rows()
    assert table_row_count > 0, f"{LANCEDB_URI}/{TABLE_NAME} is empty."

    candidate_limit = min(MAX_RETRIEVAL_CANDIDATES, table_row_count)
    candidate_depth = min(INITIAL_RETRIEVAL_CANDIDATES, candidate_limit)
    retriever_embed_kwargs = {
        "model_name": EMBED_MODEL,
        "embed_model_name": EMBED_MODEL,
    }
    if INFERENCE_MODE == "local":
        retriever_embed_kwargs["local_ingest_embed_backend"] = "hf"
        retriever_run_mode = "local"
    else:
        retriever_embed_kwargs.update(
            {
                "embedding_endpoint": EMBED_INVOKE_URL,
                "api_key": require_nvidia_api_key(),
            }
        )
        retriever_run_mode = "service"

    retriever = Retriever(
        run_mode=retriever_run_mode,
        vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
        embed_kwargs=retriever_embed_kwargs,
        top_k=candidate_depth,
    )
    retrieval_results = retriever.retrieve_batch(
        questions,
        top_k=candidate_depth,
    )
    retrieval_depths = [candidate_depth] * len(questions)

    if target_unique_documents:
        validate_retrieval_metadata(retrieval_results, query_ids)

    while target_unique_documents:
        unresolved = [
            index
            for index, result in enumerate(retrieval_results)
            if len(ranked_unique_documents(result.metadata))
            < target_unique_documents
        ]
        if not unresolved or candidate_depth >= candidate_limit:
            break

        next_depth = min(candidate_depth * 2, candidate_limit)
        print(
            f"Searching {next_depth} candidates for {len(unresolved)} "
            "questions that need more unique documents..."
        )
        expanded_results = retriever.retrieve_batch(
            [questions[index] for index in unresolved],
            top_k=next_depth,
        )
        validate_retrieval_metadata(
            expanded_results,
            [query_ids[index] for index in unresolved],
        )
        for index, expanded_result in zip(unresolved, expanded_results):
            retrieval_results[index] = expanded_result
            retrieval_depths[index] = next_depth
        candidate_depth = next_depth

    retrieval_payloads = [
        {
            "metadata": result.metadata,
            "contexts": result.chunks,
            "candidates_examined": depth,
        }
        for result, depth in zip(retrieval_results, retrieval_depths)
    ]
else:
    external_by_query_id = load_external_results(
        external_results_path,
        query_ids,
    )
    retrieval_payloads = []
    for query_id in query_ids:
        hits = external_by_query_id[str(query_id)]
        retrieval_payloads.append(
            {
                "metadata": [
                    {"document_id": hit["document_id"]} for hit in hits
                ],
                "contexts": [hit["text"] for hit in hits],
                "candidates_examined": len(hits),
            }
        )
    print(
        f"Loaded external retrieval results for "
        f"{len(retrieval_payloads)} questions."
    )

recall_columns = [f"document_recall_at_{k}" for k in RETRIEVAL_CUTOFFS]
ndcg_columns = [f"document_ndcg_at_{k}" for k in RETRIEVAL_CUTOFFS]
retrieval_metric_columns = [
    column
    for pair in zip(recall_columns, ndcg_columns)
    for column in pair
]

retrieval_rows = []
for record, query_id, payload in zip(
    selected_records,
    query_ids,
    retrieval_payloads,
):
    gold_documents = record.get(document_field) if document_field else None
    ranked_documents = ranked_unique_documents(payload["metadata"])
    row = {
        "query_id": query_id,
        "question": record[query_field],
        "reference": record[answer_field],
        "gold_document": gold_documents,
        "unique_documents_retrieved": len(ranked_documents),
        "chunk_candidates_examined": payload["candidates_examined"],
        "retrieved_contexts": payload["contexts"][:GENERATION_CONTEXT_CHUNKS],
        "retrieval_metadata": payload["metadata"],
    }
    for cutoff, recall_column, ndcg_column in zip(
        RETRIEVAL_CUTOFFS,
        recall_columns,
        ndcg_columns,
    ):
        row[recall_column] = document_recall_at_k(
            ranked_documents,
            gold_documents,
            cutoff,
        )
        row[ndcg_column] = document_ndcg_at_k(
            ranked_documents,
            gold_documents,
            cutoff,
        )
    retrieval_rows.append(row)

retrieval_df = pd.DataFrame(retrieval_rows)

if document_field:
    retrieval_summary_df = pd.DataFrame(
        {
            "Recall": [retrieval_df[column].mean() for column in recall_columns],
            "nDCG": [retrieval_df[column].mean() for column in ndcg_columns],
        },
        index=pd.Index(RETRIEVAL_CUTOFFS, name="k"),
    )
    display(
        retrieval_summary_df.style
        .format({"Recall": "{:.3f}", "nDCG": "{:.3f}"})
        .set_caption("Document-level retrieval metrics")
    )

    insufficient = (
        retrieval_df["unique_documents_retrieved"]
        < target_unique_documents
    ).sum()
    if insufficient:
        print(
            f"Warning: {insufficient} questions had fewer than "
            f"{target_unique_documents} identifiable unique documents after "
            f"examining at most {MAX_RETRIEVAL_CANDIDATES} candidates. "
            "Check source identifiers and the number of ranked results."
        )
else:
    print("No document_field configured; document metrics were skipped.")

print(f"Retrieved contexts for {len(retrieval_df)} questions.")


## 6. Generate Answers (Optional)

When `RUN_GENERATION=True`, generate one answer per question with the NVIDIA OpenAI-compatible endpoint. Model thinking is disabled, and the same `strip_think_tags()` parser used by NRL removes any visible reasoning before scoring. Thinking-only or truncated reasoning is recorded as a generation failure. The OpenAI SDK retries transient connection, timeout, rate-limit, and server failures up to five times. Final failures are retained in the report.


In [ ]:
from nemo_retriever.models.llm.text_utils import strip_think_tags
from openai import OpenAI


def generation_messages(question, contexts):
    numbered_contexts = "\n\n".join(
        f"[{index}] {context}"
        for index, context in enumerate(contexts, start=1)
    )
    return [
        {
            "role": "system",
            "content": (
                "Answer the question using only the retrieved context. "
                "Give a concise, direct answer. If the context does not "
                "contain the answer, say that the answer is not available."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Retrieved context:\n{numbered_contexts}"
            ),
        },
    ]


def format_generation_error(error):
    details = [f"{type(error).__name__}: {error}"]
    status_code = getattr(error, "status_code", None)
    request_id = getattr(error, "request_id", None)
    if status_code is not None:
        details.append(f"HTTP status: {status_code}")
    if request_id:
        details.append(f"request_id: {request_id}")
    return " | ".join(details)


retrieval_df["response"] = pd.NA
retrieval_df["generation_status"] = "skipped"
retrieval_df["generation_error"] = pd.NA

if RUN_GENERATION:
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=require_nvidia_api_key(),
        max_retries=5,
        timeout=120.0,
    )
    responses = []
    generation_statuses = []
    generation_errors = []

    for position, (_, row) in enumerate(retrieval_df.iterrows(), start=1):
        try:
            completion = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=generation_messages(
                    row["question"],
                    row["retrieved_contexts"],
                ),
                temperature=0.0,
                max_tokens=4096,
                extra_body={
                    "chat_template_kwargs": {"enable_thinking": False}
                },
            )
            raw_answer = completion.choices[0].message.content or ""
            answer = strip_think_tags(raw_answer)
            if not answer:
                reason = (
                    "thinking_truncated"
                    if raw_answer.strip()
                    else "empty_response"
                )
                raise ValueError(
                    f"{reason}: the model returned no scorable answer."
                )
            responses.append(answer)
            generation_statuses.append("success")
            generation_errors.append(None)
        except Exception as error:
            error_message = format_generation_error(error)
            responses.append("")
            generation_statuses.append("failed")
            generation_errors.append(error_message)
            print(
                f"Generation failed for query {row['query_id']} "
                f"(row {position}):\n{error_message}\n"
            )

        if position % 10 == 0 or position == len(retrieval_df):
            successful = generation_statuses.count("success")
            print(
                f"Attempted {position}/{len(retrieval_df)} answers; "
                f"{successful} successful."
            )

    retrieval_df["response"] = responses
    retrieval_df["generation_status"] = generation_statuses
    retrieval_df["generation_error"] = generation_errors
else:
    print("Answer generation skipped (RUN_GENERATION=False).")

evaluation_df = retrieval_df[
    retrieval_df["generation_status"] == "success"
].copy()
ragas_records = (
    evaluation_df[
        ["question", "retrieved_contexts", "response", "reference"]
    ]
    .rename(columns={"question": "user_input"})
    .to_dict("records")
)

if RUN_GENERATION:
    print(
        f"Generated {len(evaluation_df)} successful answers; "
        f"{len(retrieval_df) - len(evaluation_df)} failed."
    )


## 7. Evaluate with RAGAS (Optional)

When `RUN_JUDGING=True`, RAGAS evaluates every successfully generated answer for answer accuracy, context relevance, and response groundedness. The shared judge client is deliberately rate-limited and retries transient failures, so this section can take several minutes on the hosted trial endpoint. The coverage table reports how many answers actually received a score for each metric; missing scores are never silently excluded.


In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerAccuracy,
    ContextRelevance,
    ResponseGroundedness,
)
from ragas.run_config import RunConfig

ragas_results = None
ragas_metrics = [
    AnswerAccuracy(),
    ContextRelevance(),
    ResponseGroundedness(),
]
ragas_metric_columns = [metric.name for metric in ragas_metrics]
ragas_metric_df = pd.DataFrame(
    index=retrieval_df.index,
    columns=ragas_metric_columns,
    dtype=float,
)
judging_coverage_df = pd.DataFrame(
    columns=["Scored", "Missing", "Coverage"]
)

if RUN_JUDGING and not evaluation_df.empty:
    judge_rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,
        check_every_n_seconds=0.1,
        max_bucket_size=1,
    )
    judge_llm = ChatNVIDIA(
        model=JUDGE_MODEL,
        rate_limiter=judge_rate_limiter,
    )
    evaluation_dataset = EvaluationDataset.from_list(ragas_records)
    ragas_results = evaluate(
        dataset=evaluation_dataset,
        metrics=ragas_metrics,
        llm=LangchainLLMWrapper(judge_llm),
        run_config=RunConfig(
            timeout=180,
            max_retries=10,
            max_wait=60,
            max_workers=1,
        ),
    )

    raw_ragas_df = ragas_results.to_pandas().reset_index(drop=True)
    metric_values = raw_ragas_df.reindex(
        columns=ragas_metric_columns
    ).apply(pd.to_numeric, errors="coerce")
    ragas_metric_df.loc[
        evaluation_df.index,
        ragas_metric_columns,
    ] = metric_values.to_numpy()

    attempted = len(evaluation_df)
    scored_counts = metric_values.notna().sum()
    judging_coverage_df = pd.DataFrame(
        {
            "Scored": scored_counts,
            "Missing": attempted - scored_counts,
            "Coverage": scored_counts / attempted,
        }
    )
    judging_coverage_df.index.name = "Metric"
    print(f"Judging finished for {attempted} generated answers.")
    display(
        judging_coverage_df.style.format(
            {"Coverage": "{:.1%}"}
        )
    )
elif RUN_JUDGING:
    print("Judging skipped because no answers were generated successfully.")
else:
    print("RAGAS judging skipped (RUN_JUDGING=False).")


## 8. Analyze Results

The final section shows a per-question preview, optional RAGAS averages, and document retrieval metrics with one row per cutoff and separate **Recall** and **nDCG** columns. It works when generation or judging is disabled. `summary_df` contains the final document-retrieval summary, and `ragas_summary_df` contains any judging metrics.


In [ ]:
report_columns = [
    "query_id",
    "question",
    "reference",
    "gold_document",
    "unique_documents_retrieved",
    "chunk_candidates_examined",
    *retrieval_metric_columns,
    "response",
    "generation_status",
    "generation_error",
]
report_df = pd.concat(
    [retrieval_df[report_columns], ragas_metric_df],
    axis=1,
)

if document_field:
    summary_df = pd.DataFrame(
        {
            "Recall": [report_df[column].mean() for column in recall_columns],
            "nDCG": [report_df[column].mean() for column in ndcg_columns],
        },
        index=pd.Index(RETRIEVAL_CUTOFFS, name="k"),
    )
else:
    summary_df = pd.DataFrame(columns=["Recall", "nDCG"])
    summary_df.index.name = "k"

ragas_summary_rows = {}
if RUN_JUDGING:
    attempted = len(evaluation_df)
    for column in ragas_metric_columns:
        values = pd.to_numeric(report_df[column], errors="coerce")
        scored = int(values.notna().sum())
        ragas_summary_rows[column] = {
            "Mean score": values.mean(),
            "Scored": scored,
            "Missing": attempted - scored,
            "Coverage": scored / attempted if attempted else float("nan"),
        }
ragas_summary_df = pd.DataFrame.from_dict(
    ragas_summary_rows,
    orient="index",
)
ragas_summary_df.index.name = "Metric"

print("Per-question results:")
display(report_df.head())

if not ragas_summary_df.empty:
    print("RAGAS metrics:")
    display(
        ragas_summary_df.style.format(
            {"Mean score": "{:.3f}", "Coverage": "{:.1%}"},
            na_rep="—",
        )
    )
elif not RUN_JUDGING:
    print("RAGAS judging was skipped (RUN_JUDGING=False).")
else:
    print("No RAGAS scores were produced.")

if document_field:
    print("Document-level retrieval metrics:")
    display(
        summary_df.style.format(
            {"Recall": "{:.3f}", "nDCG": "{:.3f}"}
        )
    )
else:
    print("Document metrics were skipped because document_field is not configured.")
    display(summary_df)
